### Fall Colours with PACE OCI
Author: Skye Caplan

Last Updated: November 24th, 2025

This notebook presents a workflow for monitoring seasonal leaf colour change with PACE hyperspectral-enabled vegetation indices over multiple years.

<div class="alert alert-block alert-info">
<b>Note:</b> As of Nov. 24th 2025, this notebook requires a lot of memory (at least 7 GB) due to the individual averaging of daily 2025 data into 8D data. Should reduce once we have refined data for 2025. 
</div>

In [1]:
# Imports 
import xarray as xr 
import numpy as np 
import pandas as pd 
import glob 
import earthaccess
import dask
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask.distributed import Client
from dask import delayed
from os.path import basename
from tqdm.notebook import tqdm

extents = {"nam":(-150, -49, 20, 60), 
           "ec":(-95, -66,25, 50),}

auth = earthaccess.login(persist=True)

In [2]:
# Global or subset?
subset = True
scene = extents["nam"]

# 2025 data still NRT, so need to do the 8-day avg ourselves
# Didn't do this all in one call because the refined data still
#    has accompanying NRT data in CMR sometimes
ref_tspan = ("2025-08-20", "2025-09-30")
nrt_tspan = ("2025-10-01", "2025-12-31")

ref_results = earthaccess.search_data(
    short_name=["PACE_OCI_L3M_LANDVI"],
    temporal=ref_tspan,
    granule_name="*DAY*V3_1*0p1*",
)

nrt_results = earthaccess.search_data(
    short_name=["PACE_OCI_L3M_LANDVI_NRT"],
    temporal=nrt_tspan,
    granule_name="*DAY*V3_1*0p1*",
)

results_25 = ref_results + nrt_results
results_25

[Collection: {'ShortName': 'PACE_OCI_L3M_LANDVI', 'Version': '3.1'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180, 'SouthBoundingCoordinate': -90, 'NorthBoundingCoordinate': 90, 'EastBoundingCoordinate': 180}]}}}
 Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2025-08-20T23:59:59Z', 'BeginningDateTime': '2025-08-20T00:00:00Z'}}
 Size(MB): 33.158637046813965
 Data: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20250820.L3m.DAY.LANDVI.V3_1.0p1deg.nc'],
 Collection: {'Version': '3.1', 'ShortName': 'PACE_OCI_L3M_LANDVI'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'EastBoundingCoordinate': 180, 'SouthBoundingCoordinate': -90, 'NorthBoundingCoordinate': 90, 'WestBoundingCoordinate': -180}]}}}
 Temporal coverage: {'RangeDateTime': {'EndingDateTime': '2025-08-21T23:59:59Z', 'BeginningDateTime': '2025-08-21T00:00:00Z'}}
 Size(MB): 32.7402639389

In [ ]:
# Average the 2025 dailies into 2025 8-days, matching 2024 weeks 
start_dates = pd.to_datetime(['2024-08-20', '2024-08-28', '2024-09-05', '2024-09-13', '2024-09-21',
                              '2024-09-29', '2024-10-07', '2024-10-15', '2024-10-23', '2024-10-31',
                              '2024-11-08', '2024-11-16', '2024-11-24', '2024-12-02', '2024-12-10',
                              '2024-12-18', '2024-12-26'])



In [3]:
# Now 2024
# Pull 2024 8-day files 
tspan = ("2024-08-20", "2024-12-31")

# Search for all 8 day ds w/i desired time 
results_24 = earthaccess.search_data(
    short_name=["PACE_OCI_L3M_LANDVI"],
    temporal=tspan,
    granule_name="*8D*V3_1*0p1*",
)

paths_24 = earthaccess.open(results_24)
vis_24 = xr.open_mfdataset(paths_24,combine="nested", concat_dim="time").drop_vars("palette")

# Subset, if desired
if subset:
    vis_24 = vis_24.sel({"lat":slice(scene[3], scene[2]), 
                         "lon":slice(scene[0],scene[1])})
vis_24

QUEUEING TASKS | :   0%|          | 0/17 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/17 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/17 [00:00<?, ?it/s]

<xarray.Dataset> Size: 275MB
Dimensions:  (time: 17, lat: 400, lon: 1010)
Coordinates:
  * lat      (lat) float32 2kB 59.95 59.85 59.75 59.65 ... 20.25 20.15 20.05
  * lon      (lon) float32 4kB -149.9 -149.9 -149.8 ... -49.25 -49.15 -49.05
Dimensions without coordinates: time
Data variables:
    ndvi     (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    evi      (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    ndwi     (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    ndii     (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    cci      (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    ndsi     (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    pri      (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    cire     (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    car      (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
    mari     (time, lat, lon) float32 27MB dask.array<chunksize=(1, 212, 724), meta=np.ndarray>
Attributes: (12/62)
    product_name:                      PACE_OCI.20240820_20240827.L3m.8D.LAND...
    instrument:                        OCI
    title:                             OCI Level-3 Standard Mapped Image
    project:                           Ocean Biology Processing Group (NASA/G...
    platform:                          PACE
    source:                            satellite observations from OCI-PACE
    ...                                ...
    cdm_data_type:                     grid
    identifier_product_doi_authority:  http://dx.doi.org
    identifier_product_doi:            10.5067/PACE/OCI/L3M/LANDVI/3.1
    data_bins:                         1551219
    data_minimum:                      -11.276109
    data_maximum:                      13.818302

In [4]:
paths_24

[<File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20240820_20240827.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20240828_20240904.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20240905_20240912.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20240913_20240920.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20240921_20240928.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20240929_20241006.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20241007_20241014.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20241015_20241022.L3m.8D.LANDVI.V3_1.0p1deg.nc>,
 <File-like object S3FileSystem, ob-cumulus-prod-public/PACE_OCI.20241023_20241030.L3m.8

In [4]:
# Pull VIs and average for masking purposes, will use for 2025 data also 
# TODO: Maybe get land cover data for this 
avgs_24 = vis_24.mean(dim="time")
avgs24_masked = avgs_2024.where(np.logical_or(
    np.logical_and((mu_ndvi >= 0.2), 
                   (mu_cire >= 0.25)), 
    (np.isnan(avgs_2024.cire))), np.nan)

<xarray.Dataset> Size: 16MB
Dimensions:  (lat: 400, lon: 1010)
Coordinates:
  * lat      (lat) float32 2kB 59.95 59.85 59.75 59.65 ... 20.25 20.15 20.05
  * lon      (lon) float32 4kB -149.9 -149.9 -149.8 ... -49.25 -49.15 -49.05
Data variables:
    ndvi     (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    evi      (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    ndwi     (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    ndii     (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    cci      (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    ndsi     (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    pri      (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    cire     (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    car      (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>
    mari     (lat, lon) float32 2MB dask.array<chunksize=(212, 724), meta=np.ndarray>